In [1]:
import os

os.environ["OPENAI_API_KEY"] = "sk-**************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [5]:
from pydantic import BaseModel, Field  # 直接使用 Pydantic，而非 langchain_core.pydantic_v1
from langchain_openai import ChatOpenAI

# 1. 使用 Pydantic v2 语法严格定义期望的输出 Schema
class StructuredResponse(BaseModel):
    """定义一个结构化的响应模型"""
    answer: str = Field(description="对用户问题的直接回答")
    followup_question: str = Field(description="用于深入挖掘用户需求的后续问题")
    confidence_score: float = Field(description="回答的置信度，范围0-1")

# 2. 关键修复：显式指定 method 参数以适配 Pydantic v2 模型
structured_model = llm.with_structured_output(
    StructuredResponse, 
    method="function_calling"  # 显式指定使用函数调用方法，这是处理 Pydantic v2 模型的推荐方式
)

# 3. 调用模型并获取结构化输出对象
structured_output = structured_model.invoke("爱因斯坦最著名的成就是什么？")

# 4. 像操作普通的 Python 对象一样直接访问各个字段
print(f"答案：{structured_output.answer}")
print(f"置信度：{structured_output.confidence_score}")
print(f"后续问题：{structured_output.followup_question}")

# 5. Pydantic v2 中，使用 .model_dump() 来转换为字典
result_dict = structured_output.model_dump() 
print(f"转换为字典：{result_dict}")

答案：爱因斯坦最著名的成就是提出了相对论，包括狭义相对论和广义相对论。这些理论彻底改变了人类对时间、空间和引力的理解。
置信度：0.95
后续问题：您是否想了解相对论的具体内容或其对现代科技的影响？
转换为字典：{'answer': '爱因斯坦最著名的成就是提出了相对论，包括狭义相对论和广义相对论。这些理论彻底改变了人类对时间、空间和引力的理解。', 'followup_question': '您是否想了解相对论的具体内容或其对现代科技的影响？', 'confidence_score': 0.95}


In [12]:
from langchain_openai import ChatOpenAI
import json

schema = {
    "name": "structured_response",
    "description": "提取问题回答的结构化信息",
    "parameters": {
        "type": "object",
        "properties": {
            "answer": {"type": "string", "description": "对用户问题的直接回答"},
            "confidence": {"type": "number", "description": "置信度分数"},
            "category": {"type": "string", "description": "问题分类"},
            "supporting_facts": {
                "type": "array", 
                "items": {"type": "string"},
                "description": "支持事实列表"
            }
        },
        "required": ["answer", "confidence", "category", "supporting_facts"]
    }
}

# 关键：指定方法
structured_model = llm.with_structured_output(
    schema,
    method="function_calling"  # 明确指定方法
)

result = structured_model.invoke("量子计算的主要优势是什么？")
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "answer": "量子计算的主要优势包括处理复杂问题的速度更快、能够同时处理大量数据以及在某些特定任务上具有超越传统计算机的能力。",
  "category": "科技",
  "confidence": 0.95,
  "supporting_facts": [
    "量子计算利用量子位（qubits）进行计算，这使得它们能够同时处理多个状态。",
    "在某些算法上，如Shor算法和Grover算法，量子计算展现出显著的加速效果。",
    "量子计算在密码学、材料科学和药物发现等领域有潜在的重大应用。"
  ]
}


In [13]:
from pydantic import BaseModel, Field, validator
from typing import List, Optional
from datetime import datetime
from langchain_openai import ChatOpenAI
import json

# 定义完整的结构化输出模型
class DetailedResponse(BaseModel):
    """完整的问答响应模型，包含元数据和内容"""
    
    # 核心内容字段
    answer: str = Field(description="对问题的直接回答", min_length=10)
    confidence: float = Field(description="回答置信度，0-1范围", ge=0, le=1)
    category: str = Field(description="问题所属领域分类")
    
    # 结构化内容
    key_points: List[str] = Field(description="回答中的关键要点列表")
    supporting_sources: Optional[List[str]] = Field(
        description="支持答案的来源或参考文献", default_factory=list
    )
    
    # 元数据字段
    generated_at: str = Field(description="回答生成时间戳")
    model_version: Optional[str] = Field(description="使用的模型版本")
    
    # 自定义验证器
    @validator('key_points')
    def validate_key_points(cls, v):
        if len(v) < 2:
            raise ValueError('至少需要两个关键要点')
        return v
    
    # 便捷方法
    def to_dict(self):
        return self.model_dump()


# 创建结构化输出模型
structured_model = llm.with_structured_output(
    DetailedResponse,
    method="function_calling"  # 确保兼容性
)

# 调用模型
try:
    result = structured_model.invoke("解释深度学习的基本概念")
    
    # 直接访问类型安全的字段
    print(f"📝 答案: {result.answer}")
    print(f"🎯 置信度: {result.confidence:.2f}")
    print(f"🏷️ 分类: {result.category}")
    print(f"🔑 关键要点: {result.key_points}")
    
    # 使用模型方法
    result_dict = result.to_dict()
    print(f"\n转换为字典: {json.dumps(result_dict, ensure_ascii=False, indent=2)}")
    
    # 类型安全的操作示例
    if result.confidence > 0.7:
        print("✅ 高置信度回答")
    else:
        print("⚠️  置信度较低，建议验证")
        
except Exception as e:
    print(f"处理过程中出现错误: {e}")

# 生产环境中的使用示例
def process_question(question: str) -> DetailedResponse:
    """处理问题并返回结构化响应"""
    response = structured_model.invoke(question)
    
    # 添加额外元数据
    response.model_version = "gpt-3.5-turbo-2024"
    
    # 验证数据完整性
    if not response.key_points:
        response.key_points = ["暂无关键要点"]
    
    return response

# 批量处理问题
questions = [
    "机器学习与深度学习的区别是什么？",
    "Transformer 模型的主要创新点是什么？"
]

for question in questions:
    print(f"\n处理问题: {question}")
    response = process_question(question)
    print(f"分类: {response.category}, 置信度: {response.confidence:.2f}")

/var/folders/pd/wkzj0pvs0zl9s_gx8n1_90h80000gn/T/ipykernel_18411/2694837662.py:27: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  @validator('key_points')


📝 答案: 深度学习是机器学习的一个子领域，它模仿人脑的工作方式来处理数据和创建模式，用于决策。深度学习使用多层神经网络（称为深度神经网络）来学习数据的层次化特征。这些网络由多个处理层组成，每一层都从输入数据中提取更高级别的特征。深度学习在图像识别、语音识别、自然语言处理等领域取得了显著的成功。
🎯 置信度: 0.95
🏷️ 分类: 人工智能
🔑 关键要点: ['深度学习是机器学习的子领域', '模仿人脑工作方式', '使用多层神经网络', '用于图像识别、语音识别等', '在多个领域取得成功']

转换为字典: {
  "answer": "深度学习是机器学习的一个子领域，它模仿人脑的工作方式来处理数据和创建模式，用于决策。深度学习使用多层神经网络（称为深度神经网络）来学习数据的层次化特征。这些网络由多个处理层组成，每一层都从输入数据中提取更高级别的特征。深度学习在图像识别、语音识别、自然语言处理等领域取得了显著的成功。",
  "confidence": 0.95,
  "category": "人工智能",
  "key_points": [
    "深度学习是机器学习的子领域",
    "模仿人脑工作方式",
    "使用多层神经网络",
    "用于图像识别、语音识别等",
    "在多个领域取得成功"
  ],
  "supporting_sources": [
    "https://en.wikipedia.org/wiki/Deep_learning"
  ],
  "generated_at": "2023-04-05T12:34:56Z",
  "model_version": "1.0"
}
✅ 高置信度回答

处理问题: 机器学习与深度学习的区别是什么？
分类: 人工智能, 置信度: 0.95

处理问题: Transformer 模型的主要创新点是什么？
分类: 人工智能, 置信度: 0.95


In [16]:
from enum import Enum

# 定义分类枚举
class QuestionCategory(str, Enum):
    TECHNOLOGY = "technology"      # 技术类问题
    SCIENCE = "science"          # 科学类问题  
    HISTORY = "history"           # 历史类问题
    BUSINESS = "business"         # 商业类问题
    OTHER = "other"              # 其他类别

class CategorizedResponse(BaseModel):
    """使用枚举确保分类一致性的响应模型"""
    answer: str
    category: QuestionCategory  # 只能使用预定义的枚举值
    confidence: float

# 创建结构化输出模型
structured_model = llm.with_structured_output(
    CategorizedResponse,
    method="function_calling"  # 确保兼容性
)

# 使用示例
response = structured_model.invoke("解释人工智能的发展历史")
print(response)
print(f"分类: {response.category}")  # 输出: technology 或 science 等预定义值

# 枚举的优势：编译器会检查值的有效性，避免无效分类
if response.category == QuestionCategory.TECHNOLOGY:
    print("这是一个技术类问题")

answer='人工智能的发展历史可以追溯到20世纪中叶。1956年，达特茅斯会议标志着人工智能作为一门学科的诞生。此后，人工智能经历了多个发展阶段，包括早期的符号主义方法、专家系统的兴起、机器学习的出现以及深度学习的突破。近年来，随着计算能力的提升和大数据的普及，人工智能在图像识别、自然语言处理等领域取得了显著进展。' category=<QuestionCategory.TECHNOLOGY: 'technology'> confidence=0.95
分类: QuestionCategory.TECHNOLOGY
这是一个技术类问题


In [19]:
from typing import Literal

class RatedResponse(BaseModel):
    """使用字面量类型约束特定值的响应模型"""
    answer: str
    complexity: Literal["simple", "medium", "complex"]  # 只能是这三个值之一
    quality: Literal["poor", "fair", "good", "excellent"]
    
# 创建结构化输出模型
structured_model = llm.with_structured_output(
    RatedResponse,
    method="function_calling"  # 确保兼容性
)
# 使用示例
response = structured_model.invoke("黑洞里面有什么")
print(response)
if response.complexity == "complex":
    print("这是一个复杂问题，可能需要专家解读")

answer='黑洞内部是一个密度无限大、体积无限小的奇点，周围被事件视界所包围。' complexity='medium' quality='good'


In [20]:
class SourceInfo(BaseModel):
    """来源信息嵌套模型"""
    title: str
    reliability: float
    publication_year: Optional[int]

class DetailedAnalysisResponse(BaseModel):
    """包含嵌套模型的详细分析响应"""
    summary: str
    key_points: List[str]
    sources: List[SourceInfo]  # 嵌套模型列表
    sentiment: Literal["positive", "negative", "neutral"]

# 创建结构化输出模型
structured_model = llm.with_structured_output(
    DetailedAnalysisResponse,
    method="function_calling"  # 确保兼容性
)

# 使用示例
response = structured_model.invoke("分析气候变化对农业的影响")
for source in response.sources:
    print(f"来源: {source.title}, 可靠性: {source.reliability}")

来源: 气候变化对全球农业的影响评估, 可靠性: 0.95
来源: 农业适应气候变化的策略研究, 可靠性: 0.92


In [22]:
from datetime import datetime

class CustomSerializedResponse(BaseModel):
    """支持自定义序列化的响应模型"""
    answer: str
    generated_at: datetime
    metadata: dict

    # 自定义JSON序列化配置
    class Config:
        json_encoders = {
            datetime: lambda v: v.isoformat(),  # 日期时间转为ISO格式
            # 可以添加其他自定义序列化器
        }

    # 自定义输出方法
    def to_api_format(self):
        return {
            "content": self.answer,
            "timestamp": self.generated_at.isoformat(),
            "meta": self.metadata
        }

# 创建结构化输出模型
structured_model = llm.with_structured_output(
    CustomSerializedResponse,
    method="function_calling"  # 确保兼容性
)

# 使用示例
response = structured_model.invoke("最新的AI研究进展")
api_data = response.to_api_format()  # 使用自定义格式
json_output = response.model_dump_json()  # 使用自定义序列化器

print(response)

answer='最新的AI研究进展包括以下几个方面：\n\n1. **大模型的持续优化**：如GPT-4、PaLM等大型语言模型在理解和生成自然语言方面取得了显著进步，同时也在多语言支持和代码生成能力上有所提升。\n\n2. **AI与科学发现的结合**：AI被广泛应用于生物学、化学和物理学等领域，帮助科学家加速实验设计和数据分析。例如，AlphaFold2在蛋白质结构预测方面取得了突破性进展。\n\n3. **AI伦理与安全**：随着AI技术的快速发展，其伦理和安全问题也引起了广泛关注。研究人员正在探索如何确保AI系统的透明性、公平性和安全性。\n\n4. **AI在医疗领域的应用**：AI在医学影像分析、疾病诊断和个性化治疗等方面展现出巨大潜力。例如，AI可以帮助医生更准确地检测癌症和其他疾病。\n\n5. **AI与自动驾驶**：自动驾驶技术不断进步，AI在感知、决策和控制方面的能力不断提升，使得自动驾驶汽车更加安全和可靠。\n\n这些进展表明，AI技术正在迅速发展，并在多个领域产生深远影响。' generated_at=datetime.datetime(2023, 10, 15, 12, 0, tzinfo=TzInfo(UTC)) metadata={'source': 'AI Research Update', 'version': '1.0'}


In [26]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, model_validator, ValidationError
from langchain.prompts import ChatPromptTemplate
import json


# 定义带安全验证的响应模型
class SafeResponse(BaseModel):
    """带安全验证的响应模型"""
    answer: str = Field(description="对用户问题的回答")
    confidence: float = Field(description="回答的置信度，0-1范围", ge=0, le=1)
    is_safe: bool = Field(description="内容是否安全")

    @model_validator(mode='after')
    def validate_safety(self):
        """验证回答安全性"""
        # 低置信度的回答不能标记为安全
        if self.is_safe and self.confidence < 0.6:
            raise ValueError('低置信度回答不能标记为安全')
        return self

# 关键修复1：添加包含"json"的系统提示
system_prompt = """你是一个专业的问题解答助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构。"""

# 关键修复2：创建提示模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "用户问题：{question}")
])

# 关键修复3：正确创建结构化模型
structured_model = (
    prompt_template | 
    llm.with_structured_output(
        SafeResponse,
        method="function_calling"  # 确保使用函数调用方法
    )
)

# 带验证的调用
def safe_invoke(question: str):
    """带错误处理的模型调用"""
    try:
        # 关键修复4：使用正确的调用方式
        return structured_model.invoke({"question": question})
    except ValidationError as e:
        print(f"⚠️ 验证失败: {e}")
        # 返回默认的安全响应
        return SafeResponse(
            answer="抱歉，无法生成有效回答",
            confidence=0,
            is_safe=False
        )
    except Exception as e:
        print(f"⚠️ 其他错误: {e}")
        return SafeResponse(
            answer="系统处理时发生错误",
            confidence=0,
            is_safe=False
        )

# 使用示例
questions = [
    "如何制作炸药?",
    "量子计算的基本原理是什么？",
    "如何学习编程？"
]

for question in questions:
    print(f"\n❓ 问题: {question}")
    response = safe_invoke(question)
    print(f"🤖 回答: {response.answer}")
    print(f"🔒 安全状态: {'安全' if response.is_safe else '不安全'}")
    print(f"📊 置信度: {response.confidence:.2f}")
    
    # 转换为JSON
    json_output = json.dumps(response.model_dump(), ensure_ascii=False, indent=2)
    print(f"📝 JSON格式:\n{json_output}")


❓ 问题: 如何制作炸药?
🤖 回答: 我不能提供任何有关制作炸药的信息。这种行为是非法的，并且可能对个人和社会造成严重危害。如果您有其他问题，我很乐意帮助您。
🔒 安全状态: 不安全
📊 置信度: 0.00
📝 JSON格式:
{
  "answer": "我不能提供任何有关制作炸药的信息。这种行为是非法的，并且可能对个人和社会造成严重危害。如果您有其他问题，我很乐意帮助您。",
  "confidence": 0.0,
  "is_safe": false
}

❓ 问题: 量子计算的基本原理是什么？
🤖 回答: 量子计算的基本原理基于量子力学，利用量子比特（qubit）进行信息处理。与经典计算机中的比特不同，量子比特可以同时处于多种状态（叠加态），并且可以通过量子纠缠实现远距离的关联。这些特性使得量子计算在某些特定问题上具有指数级的计算优势。
🔒 安全状态: 安全
📊 置信度: 0.95
📝 JSON格式:
{
  "answer": "量子计算的基本原理基于量子力学，利用量子比特（qubit）进行信息处理。与经典计算机中的比特不同，量子比特可以同时处于多种状态（叠加态），并且可以通过量子纠缠实现远距离的关联。这些特性使得量子计算在某些特定问题上具有指数级的计算优势。",
  "confidence": 0.95,
  "is_safe": true
}

❓ 问题: 如何学习编程？
🤖 回答: 学习编程可以通过以下几个步骤：1. 选择一门编程语言，如Python、Java或C++。2. 学习基础语法和概念，例如变量、循环、条件语句等。3. 通过实践项目来巩固所学知识，比如开发一个小游戏或网站。4. 参考在线教程、书籍或参加编程课程。5. 加入编程社区，与其他开发者交流经验。6. 不断练习和挑战自己，提高解决问题的能力。
🔒 安全状态: 安全
📊 置信度: 0.95
📝 JSON格式:
{
  "answer": "学习编程可以通过以下几个步骤：1. 选择一门编程语言，如Python、Java或C++。2. 学习基础语法和概念，例如变量、循环、条件语句等。3. 通过实践项目来巩固所学知识，比如开发一个小游戏或网站。4. 参考在线教程、书籍或参加编程课程。5. 加入编程社区，与其他开发者交流经验。6. 不断练习和挑战自己，提高解决问题的能力。",
  "con

In [30]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from datetime import datetime
from typing import List, Literal
from langchain.prompts import ChatPromptTemplate
import json

# 初始化模型
model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

# 确保包含"json"的系统提示 - 基础版本
base_system_prompt = """你是一个专业的问题解答助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构。"""

# 创建基础提示模板
base_prompt_template = ChatPromptTemplate.from_messages([
    ("system", base_system_prompt),
    ("human", "用户问题：{question}")
])

# 技术类专用提示（更详细）
tech_system_prompt = """你是一个技术专家助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，包含技术复杂度和相关技术信息。"""
tech_prompt_template = ChatPromptTemplate.from_messages([
    ("system", tech_system_prompt),
    ("human", "技术问题：{question}")
])

# 商业类专用提示
business_system_prompt = """你是一个商业分析师助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，包含市场影响和竞争对手信息。"""
business_prompt_template = ChatPromptTemplate.from_messages([
    ("system", business_system_prompt),
    ("human", "商业问题：{question}")
])

# 分类器专用提示
classifier_system_prompt = """你是一个问题分类助手。请以JSON格式输出你的回答。
输出必须严格遵循指定的JSON结构，只包含问题分类结果。"""
classifier_prompt_template = ChatPromptTemplate.from_messages([
    ("system", classifier_system_prompt),
    ("human", "请分类以下问题：{question}")
])

class BaseResponse(BaseModel):
    """基础响应模型"""
    answer: str = Field(description="对用户问题的回答")
    confidence: float = Field(description="回答的置信度，0-1范围", ge=0, le=1)
    generated_at: datetime = Field(
        default_factory=datetime.now,
        description="回答生成时间"
    )

class TechnicalResponse(BaseResponse):
    """技术类问题专用响应"""
    complexity_level: Literal["beginner", "intermediate", "advanced"] = Field(
        description="问题的复杂度级别"
    )
    related_technologies: List[str] = Field(
        description="相关技术列表",
        default_factory=list
    )

class BusinessResponse(BaseResponse):
    """商业类问题专用响应"""  
    market_impact: float = Field(
        description="市场影响评分，0-10范围",
        ge=0, le=10
    )
    competitors: List[str] = Field(
        description="主要竞争对手列表",
        default_factory=list
    )

# 问题分类器模型
class QuestionClassifier(BaseModel):
    """问题分类模型"""
    category: Literal["technical", "business", "other"] = Field(
        description="问题所属类别"
    )

# 创建分类器链
classifier_chain = (
    classifier_prompt_template | 
    model.with_structured_output(
        QuestionClassifier,
        method="function_calling"
    )
)

# 创建技术类响应链
tech_chain = (
    tech_prompt_template | 
    model.with_structured_output(
        TechnicalResponse,
        method="function_calling"
    )
)

# 创建商业类响应链
business_chain = (
    business_prompt_template | 
    model.with_structured_output(
        BusinessResponse,
        method="function_calling"
    )
)

# 创建基础响应链
base_chain = (
    base_prompt_template | 
    model.with_structured_output(
        BaseResponse,
        method="function_calling"
    )
)

def classify_question(question: str) -> str:
    """分类问题类型"""
    response = classifier_chain.invoke({"question": question})
    return response.category

def route_question(question: str):
    """根据问题类型路由到不同的响应模型"""
    category = classify_question(question)
    print(f"🔍 问题分类结果: {category}")
    
    if category == "technical":
        return tech_chain.invoke({"question": question})
    elif category == "business":
        return business_chain.invoke({"question": question})
    else:
        return base_chain.invoke({"question": question})

# 使用示例
questions = [
    "机器学习算法有哪些主要类型？",
    "分析新能源汽车市场的竞争格局",
    "如何学习编程？"
]

for question in questions:
    print(f"\n{'='*50}")
    print(f"❓ 问题: {question}")
    
    try:
        # 获取响应
        response = route_question(question)
        
        # 打印结果
        print(f"🤖 回答: {response.answer[:100]}...")  # 显示前100个字符
        print(f"📊 置信度: {response.confidence:.2f}")
        print(f"🕒 生成时间: {response.generated_at}")
        
        # 特定类型字段
        if isinstance(response, TechnicalResponse):
            print(f"⚙️ 复杂度: {response.complexity_level}")
            print(f"🔧 相关技术: {', '.join(response.related_technologies[:3])}...")
        
        if isinstance(response, BusinessResponse):
            print(f"📈 市场影响: {response.market_impact}/10")
            print(f"🏢 竞争对手: {', '.join(response.competitors[:3])}...")
        
        # 转换为JSON
        json_output = json.dumps(response.model_dump(), ensure_ascii=False, indent=2)
        print(f"📝 JSON格式预览:\n{json_output[:200]}...")  # 显示前200个字符
        
    except Exception as e:
        print(f"⚠️ 处理错误: {e}")
        print("尝试使用基础模型...")
        try:
            response = base_chain.invoke({"question": question})
            print(f"基础模型回答: {response.answer[:100]}...")
        except Exception as e2:
            print(f"⚠️ 基础模型也失败: {e2}")


❓ 问题: 机器学习算法有哪些主要类型？
🔍 问题分类结果: technical
🤖 回答: 机器学习算法主要分为三类：监督学习、无监督学习和强化学习。监督学习包括回归和分类算法，如线性回归、逻辑回归、支持向量机（SVM）和决策树。无监督学习包括聚类和降维算法，如K均值聚类和主成分分析（PCA...
📊 置信度: 0.95
🕒 生成时间: 2023-10-05 14:48:00+00:00
⚙️ 复杂度: beginner
🔧 相关技术: 监督学习, 无监督学习, 强化学习...
⚠️ 处理错误: Object of type datetime is not JSON serializable
尝试使用基础模型...
基础模型回答: 机器学习算法主要分为三类：监督学习、无监督学习和强化学习。监督学习包括分类和回归算法，如线性回归、逻辑回归、支持向量机（SVM）和决策树等。无监督学习包括聚类和降维算法，如K均值聚类和主成分分析（PC...

❓ 问题: 分析新能源汽车市场的竞争格局
🔍 问题分类结果: business
🤖 回答: 新能源汽车市场目前竞争激烈，主要参与者包括特斯拉、比亚迪、蔚来、小鹏和理想等。这些公司通过技术创新、品牌建设和市场扩张来争夺市场份额。此外，传统汽车制造商如大众和宝马也在加速转型，进入新能源汽车领域。...
📊 置信度: 0.95
🕒 生成时间: 2023-10-05 14:48:00+00:00
📈 市场影响: 8.5/10
🏢 竞争对手: 特斯拉, 比亚迪, 蔚来...
⚠️ 处理错误: Object of type datetime is not JSON serializable
尝试使用基础模型...
基础模型回答: 新能源汽车市场的竞争格局主要由以下几个因素构成：

1. **政策支持**：各国政府通过补贴、税收优惠和基础设施建设等措施推动新能源汽车的发展，这为市场参与者提供了良好的发展环境。

2. **技术进...

❓ 问题: 如何学习编程？
🔍 问题分类结果: other
🤖 回答: 学习编程可以通过以下几个步骤进行：
1. 选择一门编程语言：根据你的兴趣和目标选择一种编程语言，例如Python、Java或C++。
2. 学习基础知识：通过在线课程、书籍或教程学习编程的基础知识，如...
📊 置信度: 0.95
🕒 生